In [1]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats

import dask.dataframe as dd
from pathlib import Path
import glob

import datetime as dt

import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches

In [2]:
import sys

sys.path.append("../src")
sys.path.append("../src/activity")

In [3]:
import subsampling as ss
import activity.activity_assembly as actvt
from core import SITE_NAMES, FREQ_GROUPS

from cli import get_file_paths
import plot
import pipeline

In [4]:
import suncalc

import activity.activity_assembly as actvt

from core import DC_COLOR_MAPPINGS, SEATTLE_LATITUDE, SEATTLE_LONGITUDE

In [5]:
avail = np.arange(0, 720, 6) + 6
reset_24 = avail[np.where((24*60 % avail) == 0)[0]]
reset_24

array([  6,  12,  18,  24,  30,  36,  48,  60,  72,  90,  96, 120, 144,
       180, 240, 288, 360, 480, 720])

In [6]:
cycle_lengths = [10]
percent_ons = [1/2]
specific_dc_tag = "30of30"

data_params = dict()
data_params["year"] = '2022'
data_params["cycle_lengths"] = cycle_lengths
data_params["percent_ons"] = percent_ons
dc_tags = ss.get_list_of_dc_tags(data_params["cycle_lengths"], data_params["percent_ons"])
data_params["dc_tags"] = dc_tags
data_params["cur_dc_tag"] = specific_dc_tag
data_params['detector_tag'] = 'bd2'
data_params['bin_size'] = '30'
data_params['SNR_threshold'] = 3
data_params['det_prob_threshold'] = 0.35
data_params['recording_start'] = '00:00'
data_params['recording_end'] = '16:00'
data_params['assembly_type'] = 'kmeans'

pipeline_params = dict()
pipeline_params['assemble_location_summary'] = False
pipeline_params["read_csv"] = False
pipeline_params["save_activity_grid"] = False
pipeline_params["save_presence_grid"] = False
pipeline_params["save_dc_night_comparisons"] = False
pipeline_params["save_activity_dc_comparisons"] = True
pipeline_params["save_presence_dc_comparisons"] = True
pipeline_params["show_plots"] = True
pipeline_params["show_PST"] = True

In [7]:
site_key = 'Carp'
type_key = ''
data_params["site_name"] = SITE_NAMES[site_key]
data_params["site_tag"] = site_key
data_params["type_tag"] = type_key

file_paths = get_file_paths(data_params)
activity_dets_arr = pipeline.run_for_dets(data_params, pipeline_params, file_paths)
activity_df1 = actvt.construct_activity_grid_for_number_of_dets(activity_dets_arr, data_params["cur_dc_tag"])

site_key = 'Telephone'
type_key = ''
data_params["site_name"] = SITE_NAMES[site_key]
data_params["site_tag"] = site_key
data_params["type_tag"] = type_key

file_paths = get_file_paths(data_params)
activity_dets_arr = pipeline.run_for_dets(data_params, pipeline_params, file_paths)
activity_df2 = actvt.construct_activity_grid_for_number_of_dets(activity_dets_arr, data_params["cur_dc_tag"])

In [8]:
vals = []
for val in activity_df1.index:
    vals += [f'2022-07-17 {val}']
activity_times = pd.DatetimeIndex(vals).tz_localize('UTC')
activity_dates = pd.DatetimeIndex(activity_df1.columns).strftime("%m/%d/%y")
ylabel = 'PST'
midnight = '07:00'
if ylabel=='PST':
    activity_times = activity_times.tz_convert(tz='US/Pacific')
    midnight = '00:00'
activity_plot_times = activity_times.strftime("%H:%M")
plot_times = [''] * len(activity_plot_times)
plot_times[2::6] = activity_plot_times[2::6]

activity_times = activity_times.strftime("%H:%M")
plot_dates = [''] * len(activity_dates)
plot_dates[::14] = activity_dates[::14]
dates_for_sunrise_sunset = pd.to_datetime(activity_df1.columns.values, format='%m/%d/%y')
activity_lat = [SEATTLE_LATITUDE]*len(dates_for_sunrise_sunset)
activity_lon = [SEATTLE_LONGITUDE]*len(dates_for_sunrise_sunset)
sunrise_time = pd.DatetimeIndex(suncalc.get_times(dates_for_sunrise_sunset, activity_lon, activity_lat)['sunrise_end'])
sunset_time = pd.DatetimeIndex(suncalc.get_times(dates_for_sunrise_sunset, activity_lon, activity_lat)['sunset_start'])
sunrise_seconds_from_midnight = sunrise_time.hour * 3600 + sunrise_time.minute*60 + sunrise_time.second
sunset_seconds_from_midnight = sunset_time.hour * 3600 + sunset_time.minute*60 + sunset_time.second
activity_df3 = activity_df2.reindex(columns=activity_df1.columns, fill_value=np.NaN)

In [9]:
cmap = plt.get_cmap('viridis')
norm = colors.LogNorm(vmin=1, vmax=10e3)
cmap.set_bad(color='darkred')
plt.rcParams.update({'font.size': (1.2*len(activity_dates) + 1.2*len(activity_times))})
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(1*len(activity_dates), 2*len(activity_times)), sharex=True, sharey=True)
masked_array_for_nodets = np.ma.masked_where(activity_df1.values==np.NaN, activity_df1.values)
im1 = ax1.imshow(1+(masked_array_for_nodets), cmap=cmap, norm=norm)
ax1.plot(np.arange(0, len(plot_dates)), ((sunset_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunset')
ax1.axhline(y=np.where(activity_times==midnight)[0]-0.5, linewidth=0.5*len(activity_times), linestyle='dashed', color='white', label='Midnight 0:00 PST')
ax1.plot(np.arange(0, len(plot_dates)), ((sunrise_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunrise')
ax1.grid(which='both', linewidth=6, alpha=0.4)
ax1.set_ylabel(f'Time (HH:MM, {ylabel})')
ax1.set_yticks(np.arange(0, len(plot_times))-0.5)
ax1.set_yticklabels(plot_times, ha='right')

masked_array_for_nodets = np.ma.masked_where(activity_df3.values==np.NaN, activity_df3.values)
im2 = ax2.imshow(1+masked_array_for_nodets, cmap=cmap, norm=norm)
ax2.plot(np.arange(0, len(plot_dates)), ((sunset_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunset')
ax2.axhline(y=np.where(activity_times==midnight)[0]-0.5, linewidth=0.5*len(activity_times), linestyle='dashed', color='white', label='Midnight 0:00 PST')
ax2.plot(np.arange(0, len(plot_dates)), ((sunrise_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunrise')
ax2.set_ylabel(f'Time (HH:MM, {ylabel})')
ax2.set_xlabel('Date (MM/DD/YY)')
ax2.grid(which='both', linewidth=6, alpha=0.4)
ax2.set_xticks(np.arange(0, len(activity_df1.columns))-0.5)
ax2.set_xticklabels(plot_dates, rotation=30)
ax2.set_yticks(np.arange(0, len(plot_times)) - 0.5)
ax2.set_yticklabels(plot_times, ha='right')

fig.tight_layout()

pos1 = ax1.get_position(original=False)
pos2 = ax2.get_position(original=False)
colorbar_ax = fig.add_axes([pos2.x1 + 0.02, pos2.y0, 0.02, (pos1.y1 - pos2.y0)])
cbar = fig.colorbar(im1, cax=colorbar_ax, orientation='vertical')
fig.text(x=pos1.x1 - 0.04, y=pos1.y1 + 0.04, s='Number of calls', fontweight='bold',
         fontsize=plt.rcParams['font.size']+(0.2*len(activity_dates) + 0.2*len(activity_times)))

plt.show()

In [133]:
vals = []
for val in activity_df1.index:
    vals += [f'2022-07-17 {val}']
activity_times = pd.DatetimeIndex(vals).tz_localize('UTC')
activity_dates = pd.DatetimeIndex(activity_df1.columns).strftime("%m/%d/%y")
ylabel = 'PST'
midnight = '07:00'
if ylabel=='PST':
    activity_times = activity_times.tz_convert(tz='US/Pacific')
    midnight = '00:00'
activity_plot_times = activity_times.strftime("%-H")
plot_times = [''] * len(activity_plot_times)
plot_times[2::6] = activity_plot_times[2::6]

activity_times = activity_times.strftime("%H:%M")
plot_dates = [''] * len(activity_dates)
plot_dates[::14] = activity_dates[::14]
dates_for_sunrise_sunset = pd.to_datetime(activity_df1.columns.values, format='%m/%d/%y')
activity_lat = [SEATTLE_LATITUDE]*len(dates_for_sunrise_sunset)
activity_lon = [SEATTLE_LONGITUDE]*len(dates_for_sunrise_sunset)
sunrise_time = pd.DatetimeIndex(suncalc.get_times(dates_for_sunrise_sunset, activity_lon, activity_lat)['sunrise_end'])
sunset_time = pd.DatetimeIndex(suncalc.get_times(dates_for_sunrise_sunset, activity_lon, activity_lat)['sunset_start'])
sunrise_seconds_from_midnight = sunrise_time.hour * 3600 + sunrise_time.minute*60 + sunrise_time.second
sunset_seconds_from_midnight = sunset_time.hour * 3600 + sunset_time.minute*60 + sunset_time.second
activity_df3 = activity_df2.reindex(columns=activity_df1.columns, fill_value=np.NaN)

In [134]:
cmap = plt.get_cmap('viridis')
norm = colors.LogNorm(vmin=1, vmax=10e3)
cmap.set_bad(color='darkred')
plt.rcParams.update({'font.size': (1.2*len(activity_dates) + 1.2*len(activity_times))})
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(1*len(activity_dates), 2*len(activity_times)), sharex=True, sharey=True)
masked_array_for_nodets = np.ma.masked_where(activity_df1.values==np.NaN, activity_df1.values)
im1 = ax1.imshow(1+(masked_array_for_nodets), cmap=cmap, norm=norm)
ax1.plot(np.arange(0, len(plot_dates)), ((sunset_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunset')
ax1.axhline(y=np.where(activity_times==midnight)[0]-0.5, linewidth=0.5*len(activity_times), linestyle='dashed', color='white', label='Midnight 0:00 PST')
ax1.plot(np.arange(0, len(plot_dates)), ((sunrise_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunrise')
ax1.grid(which='both', linewidth=6, alpha=0.4)
ax1.set_ylabel(f'Time (hour, {ylabel})')
ax1.set_yticks(np.arange(0, len(plot_times))-0.5)
ax1.set_yticklabels(plot_times, ha='right')

masked_array_for_nodets = np.ma.masked_where(activity_df3.values==np.NaN, activity_df3.values)
im2 = ax2.imshow(1+masked_array_for_nodets, cmap=cmap, norm=norm)
ax2.plot(np.arange(0, len(plot_dates)), ((sunset_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunset')
ax2.axhline(y=np.where(activity_times==midnight)[0]-0.5, linewidth=0.5*len(activity_times), linestyle='dashed', color='white', label='Midnight 0:00 PST')
ax2.plot(np.arange(0, len(plot_dates)), ((sunrise_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunrise')
ax2.set_ylabel(f'Time (hour, {ylabel})')
ax2.set_xlabel('Date (MM/DD/YY)')
ax2.grid(which='both', linewidth=6, alpha=0.4)
ax2.set_xticks(np.arange(0, len(activity_df1.columns))-0.5)
ax2.set_xticklabels(plot_dates, rotation=30)
ax2.set_yticks(np.arange(0, len(plot_times)) - 0.5)
ax2.set_yticklabels(plot_times, ha='right')

fig.tight_layout()

pos1 = ax1.get_position(original=False)
pos2 = ax2.get_position(original=False)
colorbar_ax = fig.add_axes([pos2.x1 + 0.02, pos2.y0, 0.02, (pos1.y1 - pos2.y0)])
cbar = fig.colorbar(im1, cax=colorbar_ax, orientation='vertical')
fig.text(x=pos1.x1 - 0.04, y=pos1.y1 + 0.04, s='Number of calls', fontweight='bold',
         fontsize=plt.rcParams['font.size']+(0.2*len(activity_dates) + 0.2*len(activity_times)))

plt.show()